## Load the language model; set the GPU

In [ ]:
import os
from nnsight import LanguageModel

os.environ["CUDA_VISIBLE_DEVICES"] = "7"

model = LanguageModel(
    "Qwen/Qwen3-32B",
    attn_implementation="eager"
)
model.to_empty(device="cuda:0")
model.device

/disk/u/gio/.conda/envs/rle/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda', index=0)

## Load the problem

In [12]:
import json

dataset = []
with open("../data/hsp_step_accuracies.jsonl", 'r') as f:
    for line in f:
        dataset.append(json.loads(line))

problem1 = dataset[0]

essp_step_accuracies = problem1['essp_step_accuracies'][:-1]
hsp_step_accuracies = problem1['hsp_step_accuracies']

essp_index = problem1['first_essp_index']

cot_up_to_essp = "\n".join(problem1['steps'][:essp_index+1])

print(cot_up_to_essp)

Okay, so I need to figure out how many endpoints Figure 5 will have if we continue the pattern shown.
Let me start by understanding the pattern from the given figures.
First, let me look at Figure 1.
It's a single line segment with three segments coming out of the top, forming a Y shape.
Wait, actually, looking at the Asymptote code, Figure 1 is drawn with three line segments: from (0,0) down to (0,-3), then from (0,0) to (-2,2) and (0,0) to (2,2).
So, it's like a central point with three lines coming out: one straight down, and two going up to the sides.
So, the endpoints here would be the tips of these three lines.
The bottom tip is at (0,-3), and the two upper tips are at (-2,2) and (2,2).
So, Figure 1 has 3 endpoints.
Now, moving to Figure 2.
The Asymptote code draws a more complex figure.
Let me try to visualize it.
The description says each line-segment extremity is replaced by a gradually smaller Y.
So, in Figure 1, each endpoint is replaced by a Y in Figure 2.
Let me count the 

## Build the prompt

In [25]:
tokenizer = model.tokenizer

SYSTEM_PROMPT = (
    "You are a helpful reasoning assistant. "
    "Output your final answer inside \\boxed{}."
)

def build_base_prompt(problem):
    """Build the base prompt for a problem."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


problem = problem1['problem']

base_prompt = build_base_prompt(problem)

prompt = f"{base_prompt}<think>\n{cot_up_to_essp}"
prompt += "\n</think>\nTherefore, the final answer is \\boxed{"

print(prompt)

<|im_start|>system
You are a helpful reasoning assistant. Output your final answer inside \boxed{}.<|im_end|>
<|im_start|>user
If you continue this pattern in which each line-segment extremity is replaced by a gradually smaller Y in the next figure, in the manner shown, how many endpoints will Figure 5 have?

[asy]
draw((0,0)--(0,-3),linewidth(.75));
draw((0,0)--(-2,2),linewidth(.75));
draw((0,0)--(2,2),linewidth(.75));
label("Figure 1",(0,-3),S);

draw((5,0)--(5,-2),linewidth(.75));
draw((4,-3)--(5,-2)--(6,-3),linewidth(.75));
draw((4,1)--(5,0)--(6,1),linewidth(.75));
draw((3,1)--(4,1)--(4,2),linewidth(.75));
draw((6,2)--(6,1)--(7,1),linewidth(.75));
label("Figure 2",(5,-3),S);

draw((10,0)--(10,-2),linewidth(.75));
draw((9.5,-2.5)--(10,-2)--(10.5,-2.5),linewidth(.75));
draw((9,-2.5)--(9.5,-2.5)--(9.5,-3),linewidth(.75));
draw((11,-2.5)--(10.5,-2.5)--(10.5,-3),linewidth(.75));

draw((9,1)--(10,0)--(11,1),linewidth(.75));
draw((8.5,1)--(9,1)--(9,1.5),linewidth(.75));
draw((11.5,1)--(11

## Integrated Gradient Function

In [26]:
import torch
from tqdm import trange

def integrated_gradients(
    model: LanguageModel,
    input_text: str,
    target_token_id: int, # Which output token to attribute
    baseline_id: int | None = None,
    interpolation_steps: int = 50
):
    if baseline_id is None:
        baseline_id = model.tokenizer.pad_token_id or model.tokenizer.eos_token_id
        
    # Get the baseline embedding
    baseline_embed = model.model.embed_tokens.weight[baseline_id].detach()
    
    # Tokenize the full input
    token_ids = model.tokenizer.encode(input_text, add_special_tokens=False)
    tokens = model.tokenizer.tokenize(input_text)
    
    # Get all token embeddings: shape (seq_len, hidden_dim)
    token_embeds = model.model.embed_tokens.weight[token_ids].detach()
    
    # Baseline is repeated for each position: shape (seq_len, hidden_dim)
    baseline_embeds = baseline_embed.unsqueeze(0).expand_as(token_embeds)
    
    # Difference between input and baseline
    x_minus_baseline = token_embeds - baseline_embeds # (seq_len, hidden_dim)
    
    # Accumulate gradients across interpolation steps
    accumulated_grads = torch.zeros_like(token_embeds).to(device="cpu")
    print(accumulated_grads.device)
    
    for step in trange(1, interpolation_steps + 1):
        alpha = step / interpolation_steps
        interpolated_embeds = baseline_embeds + alpha * x_minus_baseline
        
        with model.trace(input_text):
            # Move to correct device and add batch dimension INSIDE trace
            interpolated_embeds_traced = interpolated_embeds.unsqueeze(0).to(model.device).requires_grad_(True)
            
            # Override the embedding output
            model.model.embed_tokens.output = interpolated_embeds_traced
            
            # Get logits
            logits = model.output[0]
            target_logit = logits[0, -1, target_token_id]
            
            # Compute gradients
            target_logit.backward()
            grad = interpolated_embeds_traced.grad.save()
            
        print(f"{grad.device=}")
        
        accumulated_grads += grad.squeeze(0)
        
    # Average the gradients
    avg_grads = accumulated_grads / interpolation_steps
    
    print(avg_grads.device)
    
    print(x_minus_baseline.device)
    
    x_minus_baseline = x_minus_baseline.to(device='cpu')
    
    print(x_minus_baseline.device)
    
    # Integrated gradients = (x - x') * avg_grads
    ig_attributions = x_minus_baseline * avg_grads # (seq_len, hidden_dim)
    
    # sum across hidden dimension to get per-token attributino
    token_attributions = ig_attributions.sum(dim=-1) # (seq_len,)
    
    return {
        'tokens': tokens,
        'token_ids': token_ids,
        "attributions": token_attributions,
        "attributions_full": ig_attributions,
    }

## IG Saving Function

In [27]:
def to_json_safe(obj):
    if torch.is_tensor(obj):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [to_json_safe(x) for x in obj]
    else:
        return obj

## Helper Methods

In [28]:
def extract_cot_minus_answer(full_cot):
    # Calculate the index of the last character of "\\boxed{"
    index_of_boxed = full_cot.rfind("\\boxed{", )
    index_of_answer = index_of_boxed + +len("\\boxed{")
    return full_cot[:index_of_answer]

def get_target_token_id(full_cot, cot_minus_answer):
    """Get the id of the first token of the final answer."""
    token_ids = model.tokenizer.encode(full_cot, add_special_tokens=False)
    prefix_tokens = model.tokenizer.encode(cot_minus_answer, add_special_tokens=False)
    target_token_id = token_ids[len(prefix_tokens)]
    return target_token_id

## Sanity Check

In [29]:
answer = problem1['answer']
full_cot = problem1['full_cot']
print(answer)

cot_minus_answer = extract_cot_minus_answer(full_cot)

target_id = get_target_token_id(full_cot, cot_minus_answer)
print(target_id)
print(model.tokenizer.decode(target_id))

48
19
4


In [30]:
prompt

'<|im_start|>system\nYou are a helpful reasoning assistant. Output your final answer inside \\boxed{}.<|im_end|>\n<|im_start|>user\nIf you continue this pattern in which each line-segment extremity is replaced by a gradually smaller Y in the next figure, in the manner shown, how many endpoints will Figure 5 have?\n\n[asy]\ndraw((0,0)--(0,-3),linewidth(.75));\ndraw((0,0)--(-2,2),linewidth(.75));\ndraw((0,0)--(2,2),linewidth(.75));\nlabel("Figure 1",(0,-3),S);\n\ndraw((5,0)--(5,-2),linewidth(.75));\ndraw((4,-3)--(5,-2)--(6,-3),linewidth(.75));\ndraw((4,1)--(5,0)--(6,1),linewidth(.75));\ndraw((3,1)--(4,1)--(4,2),linewidth(.75));\ndraw((6,2)--(6,1)--(7,1),linewidth(.75));\nlabel("Figure 2",(5,-3),S);\n\ndraw((10,0)--(10,-2),linewidth(.75));\ndraw((9.5,-2.5)--(10,-2)--(10.5,-2.5),linewidth(.75));\ndraw((9,-2.5)--(9.5,-2.5)--(9.5,-3),linewidth(.75));\ndraw((11,-2.5)--(10.5,-2.5)--(10.5,-3),linewidth(.75));\n\ndraw((9,1)--(10,0)--(11,1),linewidth(.75));\ndraw((8.5,1)--(9,1)--(9,1.5),linewidth

## Compute IG

In [ ]:
result = integrated_gradients(
    model=model,
    input_text=prompt,
    target_token_id=target_id,
    interpolation_steps=20
)

for token, attr in zip(result["tokens"], result["attributions"]):
    print(f"{token:>10}: {attr.item():+.4f}")

cpu


  0%|          | 0/20 [00:00<?, ?it/s]

Loading checkpoint shards: 100%|██████████| 17/17 [00:08<00:00,  1.91it/s]


## Save Result

In [ ]:
with open('../data/attributions/result2.1.json', 'w') as f:
    json.dump(to_json_safe(result), f, indent=2)